# Preparación de los dos conjuntos:

In [1]:
import os
import shutil

# Rutas base
base_dud = "../data/raw/dud"
base_acs = "../data/raw/acs/"
output_dud = "../data/process/dud/"
output_acs = "../data/process/acs/"

# Asegurar que las carpetas de salida existan
os.makedirs(output_dud, exist_ok=True)
os.makedirs(output_acs, exist_ok=True)

# Subcarpetas que se deben explorar
subdirs = ["n1", "n2"]

paired_count = 0
missing_count = 0

for subdir in subdirs:
    dud_dir = os.path.join(base_dud, subdir)
    acs_dir = os.path.join(base_acs, subdir)
    
    if not os.path.isdir(dud_dir) or not os.path.isdir(acs_dir):
        print(f"Advertencia: faltan carpetas en {subdir}")
        continue
    
    dud_files = [f for f in os.listdir(dud_dir) if f.endswith(".fits")]
    
    for dud_file in dud_files:
        # Construir el nombre correspondiente en ACS
        base_name = dud_file.replace(".fits", "")
        acs_file = base_name + "_ACS.fits"

        dud_path = os.path.join(dud_dir, dud_file)
        acs_path = os.path.join(acs_dir, acs_file)

        if os.path.exists(acs_path):
            # Copiar ambos archivos a su nueva ubicación
            shutil.copy2(dud_path, os.path.join(output_dud, dud_file))
            shutil.copy2(acs_path, os.path.join(output_acs, acs_file))
            paired_count += 1
        else:
            missing_count += 1

print(f"\nProceso completado.")
print(f"Pares encontrados y copiados: {paired_count}")
print(f"Archivos DUD sin par ACS: {missing_count}")



Proceso completado.
Pares encontrados y copiados: 16114
Archivos DUD sin par ACS: 5886


# Procesado y normalizacion de los datos para su uso por un modelo
No se ha hecho, pero estaria bien:  (Si se ha hecho una normalización global)

📉 3. Normalización estadística por imagen  

Justificación: La normalización min-max global es buena para mantener una escala uniforme, pero si hay alta variabilidad entre imágenes, podrías experimentar con:  
- Normalización por imagen individual usando media y desviación típica (z-score).  
- O bien escalar entre 0 y 1 por imagen, especialmente útil si el rango dinámico varía mucho entre ejemplos.  

In [2]:
import os
import numpy as np
from astropy.io import fits
from tqdm import tqdm

# Directorios de entrada
dud_dir = "../data/process/dud/"
acs_dir = "../data/process/acs/"

# Directorios de salida
output_base = "../data/normaliced/"
dud_output_dir = os.path.join(output_base, "dud")
acs_output_dir = os.path.join(output_base, "acs")
os.makedirs(dud_output_dir, exist_ok=True)
os.makedirs(acs_output_dir, exist_ok=True)

# Normalización min-max
def normalize_dud(x):
    return (x + 7.353157043457031) / (366.3216857910156 + 7.353157043457031)

def normalize_acs(x):
    return (x + 5.457014560699463) / (136.9979248046875 + 5.457014560699463)

# Procesar todos los archivos
dud_files = [f for f in os.listdir(dud_dir) if f.endswith(".fits")]
paired_count = 0

print("Normalizando y guardando como .npy en carpetas separadas...")
for dud_file in tqdm(dud_files):
    base_name = dud_file.replace(".fits", "")
    acs_file = base_name + "_ACS.fits"

    dud_path = os.path.join(dud_dir, dud_file)
    acs_path = os.path.join(acs_dir, acs_file)

    if not os.path.exists(acs_path):
        continue  # saltar si falta el par

    try:
        # Cargar y normalizar DUD
        with fits.open(dud_path) as dud_hdul:
            dud_data = next(hdu.data for hdu in dud_hdul if hdu.data is not None)
            dud_data = dud_data.astype(np.float32)
            dud_data = normalize_dud(dud_data)

        # Cargar y normalizar ACS
        with fits.open(acs_path) as acs_hdul:
            acs_data = next(hdu.data for hdu in acs_hdul if hdu.data is not None)
            acs_data = acs_data.astype(np.float32)
            acs_data = normalize_acs(acs_data)

        # Guardar archivos .npy en sus carpetas respectivas
        np.save(os.path.join(dud_output_dir, base_name + ".npy"), dud_data)
        np.save(os.path.join(acs_output_dir, base_name + ".npy"), acs_data)

        paired_count += 1

    except Exception as e:
        print(f"Error procesando {dud_file}: {e}")

print(f"\n✅ Total de pares procesados y guardados: {paired_count}")

Normalizando y guardando como .npy en carpetas separadas...


100%|██████████| 16114/16114 [03:01<00:00, 88.87it/s] 


✅ Total de pares procesados y guardados: 16114


# Aumento de datos

## Rotación y flip:

In [3]:
import os
import numpy as np
from tqdm import tqdm

# Directorios
input_dud_dir = "../data/normaliced/dud/"
input_acs_dir = "../data/normaliced/acs/"

output_dud_dir = "../data/Augmentation/dud/"
output_acs_dir = "../data/Augmentation/acs/"

os.makedirs(output_dud_dir, exist_ok=True)
os.makedirs(output_acs_dir, exist_ok=True)

# Aumentos (función, sufijo)
augmentations = [
    (lambda x: np.rot90(x, 1), "rot90"),
    (lambda x: np.rot90(x, 2), "rot180"),
    (lambda x: np.rot90(x, 3), "rot270"),
    (lambda x: np.flipud(x), "flipud"),
    (lambda x: np.fliplr(x), "fliplr"),
    (lambda x: np.flipud(np.rot90(x, 1)), "flipud_rot90"),
    (lambda x: np.fliplr(np.rot90(x, 1)), "fliplr_rot90"),
]

original_count = 0
augmented_count = 0

print("🔄 Generando aumentos de datos y copiando originales...")

# Buscar pares válidos
dud_files = [f for f in os.listdir(input_dud_dir) if f.endswith(".npy")]

for dud_file in tqdm(dud_files):
    base_name = dud_file.replace(".npy", "")
    acs_file = base_name + ".npy"

    dud_path = os.path.join(input_dud_dir, dud_file)
    acs_path = os.path.join(input_acs_dir, acs_file)

    if not os.path.exists(acs_path):
        print(f"⚠️ Falta el archivo ACS para {dud_file}")
        continue

    # Cargar datos
    dud_data = np.load(dud_path)
    acs_data = np.load(acs_path)

    # Guardar copia original
    np.save(os.path.join(output_dud_dir, f"{base_name}.npy"), dud_data)
    np.save(os.path.join(output_acs_dir, f"{base_name}.npy"), acs_data)
    original_count += 1

    # Aumentos
    for i, (aug_fn, suffix) in enumerate(augmentations, start=1):
        aug_dud = aug_fn(dud_data)
        aug_acs = aug_fn(acs_data)

        aug_dud_path = os.path.join(output_dud_dir, f"{base_name}_aug{i}_{suffix}.npy")
        aug_acs_path = os.path.join(output_acs_dir, f"{base_name}_aug{i}_{suffix}.npy")

        np.save(aug_dud_path, aug_dud)
        np.save(aug_acs_path, aug_acs)
        augmented_count += 1

# Estadísticas finales
total = original_count + augmented_count
print("\n✅ Aumento de datos completado.")
print(f"🔹 Pares originales: {original_count}")
print(f"🔸 Datos aumentados: {augmented_count}")
print(f"📦 Total final de archivos: {total}")


🔄 Generando aumentos de datos y copiando originales...


100%|██████████| 16114/16114 [01:58<00:00, 135.72it/s]


✅ Aumento de datos completado.
🔹 Pares originales: 16114
🔸 Datos aumentados: 112798
📦 Total final de archivos: 128912


## Zoom y deslizamiento

In [4]:
import os
import numpy as np
from tqdm import tqdm

# Directorios de entrada
input_dud_dir = "../data/Augmentation/dud/"
input_acs_dir = "../data/Augmentation/acs/"

# Directorios de salida
output_dud_dir = "../data/AugZum/dud/"
output_acs_dir = "../data/AugZum/acs/"
os.makedirs(output_dud_dir, exist_ok=True)
os.makedirs(output_acs_dir, exist_ok=True)

# Parámetros de zoom (2 píxeles de margen)
crop_margin = 2
original_count = 0
augmented_count = 0

print("🔍 Generando aumentos por zoom con deslizamiento...")

dud_files = [f for f in os.listdir(input_dud_dir) if f.endswith(".npy")]

for file in tqdm(dud_files):
    base_name = file.replace(".npy", "")
    dud_path = os.path.join(input_dud_dir, file)
    acs_path = os.path.join(input_acs_dir, file)

    if not os.path.exists(acs_path):
        print(f"⚠️ Falta el archivo ACS para {file}")
        continue

    # Cargar imágenes
    dud = np.load(dud_path)
    acs = np.load(acs_path)

    h_dud, w_dud = dud.shape
    h_acs, w_acs = acs.shape

    # Guardar copia original
    np.save(os.path.join(output_dud_dir, file), dud)
    np.save(os.path.join(output_acs_dir, file), acs)
    original_count += 1

    # 4 crops con desplazamiento de 2 píxeles
    for idx, (dy, dx) in enumerate([(0, 0), (0, crop_margin), (crop_margin, 0), (crop_margin, crop_margin)], start=1):
        dud_crop = dud[dy:h_dud - crop_margin + dy, dx:w_dud - crop_margin + dx]
        acs_crop = acs[dy:h_acs - crop_margin + dy, dx:w_acs - crop_margin + dx]

        dud_out = os.path.join(output_dud_dir, f"{base_name}_zoom{idx}.npy")
        acs_out = os.path.join(output_acs_dir, f"{base_name}_zoom{idx}.npy")

        np.save(dud_out, dud_crop)
        np.save(acs_out, acs_crop)
        augmented_count += 1

# Resumen
total = original_count + augmented_count
print("\n✅ Zoom con deslizamiento completado.")
print(f"🔹 Originales procesados: {original_count}")
print(f"🔸 Zooms generados: {augmented_count}")
print(f"📦 Total final: {total}")

🔍 Generando aumentos por zoom con deslizamiento...


100%|██████████| 128912/128912 [08:51<00:00, 242.73it/s]


✅ Zoom con deslizamiento completado.
🔹 Originales procesados: 128912
🔸 Zooms generados: 515648
📦 Total final: 644560


In [5]:
import os
import numpy as np
import torch
import torch.nn.functional as F
from tqdm import tqdm

def resize_array(array, size=(64, 64)):
    tensor = torch.tensor(array, dtype=torch.float32).unsqueeze(0).unsqueeze(0)
    resized = F.interpolate(tensor, size=size, mode='bilinear', align_corners=False)
    return resized.squeeze().numpy()

# Directorios de entrada y salida
input_dud_dir = "../data/AugZum/dud/"
input_acs_dir = "../data/AugZum/acs/"
output_dud_dir = "../data/Resized64/dud/"
output_acs_dir = "../data/Resized64/acs/"
os.makedirs(output_dud_dir, exist_ok=True)
os.makedirs(output_acs_dir, exist_ok=True)

resize_shape = (64, 64)
count = 0

for file in tqdm(os.listdir(input_dud_dir)):
    if not file.endswith(".npy"):
        continue

    dud_path = os.path.join(input_dud_dir, file)
    acs_path = os.path.join(input_acs_dir, file)
    
    if not os.path.exists(acs_path):
        continue

    try:
        dud = np.load(dud_path)
        acs = np.load(acs_path)

        dud_resized = resize_array(dud, size=resize_shape)
        acs_resized = resize_array(acs, size=resize_shape)

        np.save(os.path.join(output_dud_dir, file), dud_resized)
        np.save(os.path.join(output_acs_dir, file), acs_resized)

        count += 1
    except Exception as e:
        print(f"Error en {file}: {e}")

print(f"\nTotal de pares redimensionados: {count}")


  4%|▍         | 26161/644560 [00:36<14:43, 699.76it/s]

Error en n25_650_1036875_aug5_fliplr_zoom1.npy: Input and output sizes should be greater than 0, but got input (H: 1, W: 0) output (H: 64, W: 64)


  9%|▊         | 54956/644560 [01:22<13:22, 734.64it/s]

Error en n25_650_1036875_aug5_fliplr_zoom3.npy: Input and output sizes should be greater than 0, but got input (H: 1, W: 0) output (H: 64, W: 64)


 10%|█         | 65976/644560 [01:37<12:05, 797.63it/s]

...

Error en n25_650_1036875_aug7_fliplr_rot90_zoom1.npy: Input and output sizes should be greater than 0, but got input (H: 0, W: 1) output (H: 64, W: 64)


 98%|█████████▊| 632789/644560 [14:04<00:15, 784.49it/s]

Error en n25_650_1036875_aug2_rot180_zoom4.npy: Input and output sizes should be greater than 0, but got input (H: 1, W: 0) output (H: 64, W: 64)


100%|██████████| 644560/644560 [14:18<00:00, 750.38it/s]


Total de pares redimensionados: 644528


In [ ]:
import os
import numpy as np
from collections import Counter

def contar_shapes(folder_path):
    shapes = []
    for file in os.listdir(folder_path):
        if file.endswith(".npy"):
            path = os.path.join(folder_path, file)
            try:
                arr = np.load(path)
                shapes.append(arr.shape)
            except Exception as e:
                print(f"Error leyendo {file}: {e}")
    return Counter(shapes)

# Carpetas redimensionadas
dud_folder = "../data/Resized64/dud"
acs_folder = "../data/Resized64/acs"

print("📂 Análisis de DUD:")
conteo_dud = contar_shapes(dud_folder)
for shape, count in conteo_dud.items():
    print(f"  {shape}: {count} imágenes")

print("\n📂 Análisis de ACS:")
conteo_acs = contar_shapes(acs_folder)
for shape, count in conteo_acs.items():
    print(f"  {shape}: {count} imágenes")



📂 Análisis de DUD:
  (64, 64): 644528 imágenes

📂 Análisis de ACS:
  (64, 64): 644528 imágenes
